# The Node.js REPL

The Node.js REPL (Read-Eval-Print Loop) is a built-in interactive shell that executes JavaScript code line-by-line directly in your terminal. It functions similarly to the browser's developer console and serves as an efficient tool for testing quick code snippets, exploring Node.js modules, and debugging.

## Starting and Exiting the REPL

Open your terminal, type `node` with no arguments, and press Enter:

```bash
$ node
Welcome to Node.js v26.6.0.
Type ".help" for more information.
>
```

To exit: type `.exit`, press `Ctrl + C` twice, or press `Ctrl + D` once.

## Core Components

The acronym defines the continuous four-step cycle of the environment:

- **Read** — Takes your JavaScript input from the prompt and parses it into memory.
- **Eval** — Processes the parsed JavaScript data structure to resolve its logic.
- **Print** — Outputs the evaluated result directly onto your screen.
- **Loop** — Refreshes the command line prompt and waits for the next input line.

## Essential Tricks & Shortcuts

### The underscore variable (`_`)

`_` automatically caches the result of the last evaluated expression.

```javascript
> 10 + 20
30
> _ * 2
60
```

There's also `_error`, which holds the most recent uncaught error — handy when something throws and you want to inspect the stack.

> **Gotcha:** assigning to `_` yourself (`const _ = 5`) disables the auto-caching for the rest of the session. Node prints a warning when this happens.

### Other shortcuts

- **Multi-line blocks** — Open a block (an `if`, function, or loop) and press Enter; the REPL detects the incomplete statement and shows `...` so you can continue.
- **Command history** — `Up` / `Down` arrows cycle through previous commands. History persists across sessions in `~/.node_repl_history`.
- **Tab completion** — Type the start of a variable, object property, or core module name and press `Tab`. Pressing `Tab` on an empty prompt lists everything in scope.
- **`Ctrl + L`** — Clears the screen without losing your session state.
- **Reverse search** — `Ctrl + R` searches backwards through history, like in bash.

## Built-in Dot Commands

| Command | What it does |
| --- | --- |
| `.help` | Lists all available dot commands |
| `.break` / `.clear` | Escapes a multi-line expression you're stuck inside |
| `.editor` | Switches to a multi-line editor mode; `Ctrl + D` evaluates, `Ctrl + C` cancels |
| `.save <filepath>` | Writes the session's successfully evaluated input to a `.js` file |
| `.load <filepath>` | Reads and executes a local JavaScript file into the running session |
| `.exit` | Quits the REPL |

## Things the REPL Does That a Script Doesn't

These are the behaviours that make the REPL convenient — and that occasionally confuse people when code behaves differently once pasted into a file.

- **Core modules are pre-loaded.** You can type `fs.readdirSync('.')` or `os.cpus()` immediately, with no `require` at all. In a real script you must import them.
- **Top-level `await` works.** `> await fetch('https://example.com')` resolves at the prompt. Promises are also auto-awaited when printed, so you see the resolved value rather than `Promise { <pending> }`.
- **It runs as CommonJS.** `require()` is available; `import x from 'y'` statement syntax is **not**. Use dynamic `import('y')` instead, which returns a promise you can await:
  ```javascript
  > const { setTimeout: sleep } = await import('node:timers/promises');
  ```
- **Everything is global and sloppy-mode by default.** Redeclaring a `let` or `const` in the same session throws; if that annoys you, `var` reassignment is tolerated. Set `NODE_REPL_MODE=strict` to run in strict mode instead.

## One-Liners Without Entering the REPL

Often faster than opening an interactive session:

```bash
node -e "console.log(process.version)"    # evaluate, print nothing automatically
node -p "1 + 1"                           # evaluate AND print the result
node -p "require('os').totalmem() / 1e9"
node -r ./setup.js                        # preload a module, then run
node -i -r ./setup.js                     # preload, then drop into the REPL
```

`node --watch script.js` re-runs a file on save, which usually beats the REPL for anything longer than a few lines.

## Inspecting Deep Objects

The REPL prints objects with `util.inspect` at a depth of 2, so nested data shows up as `[Object]`. To see everything:

```javascript
> console.dir(myObject, { depth: null, colors: true });
```

You can change the default for the session by setting `util.inspect.defaultOptions.depth = null`.

## Useful Environment Variables

| Variable | Effect |
| --- | --- |
| `NODE_REPL_HISTORY` | Path to the history file; set to `""` to disable persistence |
| `NODE_REPL_HISTORY_SIZE` | Number of lines to keep (default 1000) |
| `NODE_REPL_MODE` | `sloppy` (default) or `strict` |

## Programmatic Usage

You can customize, instantiate, and embed a REPL session directly inside your own Node.js applications with the built-in `node:repl` module — useful for giving a long-running server an admin console:

```javascript
import repl from 'node:repl';

const myRepl = repl.start({
  prompt: 'my-app > ',
  useColors: true,
  ignoreUndefined: true,   // don't print `undefined` for void expressions
});

// Inject custom context variables into the REPL session
myRepl.context.db = databaseConnection;
myRepl.context.mySecretToken = 'ABC123XYZ';

// Add your own dot command
myRepl.defineCommand('stats', {
  help: 'Print server stats',
  action() {
    console.log(getStats());
    this.displayPrompt();
  },
});

// Persist history across restarts
myRepl.setupHistory('./.admin_repl_history', () => {});
```

## When Not to Use It

The REPL has no file, no version control, and no easy way to edit line 3 after you've typed line 10. It's for probing an API, checking a regex, or confirming what a method returns. For anything you'll want to run twice, write a scratch file and use `node --watch`.

## References

- [Node.js docs — How to use the REPL](https://nodejs.org/learn/command-line/how-to-use-the-nodejs-repl)
- [Node.js API — `repl` module](https://nodejs.org/api/repl.html)
- [DigitalOcean — How To Use the Node.js REPL](https://www.digitalocean.com/community/tutorials/how-to-use-the-node-js-repl)